In [ ]:
// =====================================================
// MICRO-INVERNADERO INTELIGENTE
// ESP32 - C++
// Versión 2.0.0
// =====================================================

// ---------- Pines ----------
#define PIN_TEMPERATURA 34   // LM35 - ADC
#define PIN_LDR         32   // LDR - ADC

#define PIN_VENTILADOR  18   // Motor DC - PWM
#define PIN_LED         19   // LED de potencia - PWM

// ---------- Canales PWM ----------
#define CANAL_VENTILADOR 0
#define CANAL_LED        1

// ---------- Configuración PWM ----------
#define FRECUENCIA_PWM 5000
#define RESOLUCION_PWM 8     // 8 bits: 0 - 255

// ---------- Variables ----------
float temperatura = 0.0;
int valorLDR = 0;

int dutyVentilador = 0;
int dutyLED = 0;


// =====================================================
// CONFIGURACIÓN
// =====================================================
void setup() {

  Serial.begin(115200);

  // Configuración ADC
  analogReadResolution(12);

  // Configuración PWM del ventilador
  ledcSetup(
    CANAL_VENTILADOR,
    FRECUENCIA_PWM,
    RESOLUCION_PWM
  );

  ledcAttachPin(
    PIN_VENTILADOR,
    CANAL_VENTILADOR
  );

  // Configuración PWM del LED
  ledcSetup(
    CANAL_LED,
    FRECUENCIA_PWM,
    RESOLUCION_PWM
  );

  ledcAttachPin(
    PIN_LED,
    CANAL_LED
  );

  // Iniciar actuadores apagados
  ledcWrite(CANAL_VENTILADOR, 0);
  ledcWrite(CANAL_LED, 0);

  Serial.println("=================================");
  Serial.println(" MICRO-INVERNADERO INTELIGENTE");
  Serial.println(" Sistema iniciado");
  Serial.println("=================================");
}


// =====================================================
// LECTURA DEL LM35
// =====================================================
float leerTemperatura() {

  int valorADC = analogRead(PIN_TEMPERATURA);

  // ESP32 ADC de 12 bits: 0 - 4095
  // Referencia aproximada de 3.3 V
  float voltaje = (valorADC * 3.3) / 4095.0;

  // LM35:
  // 10 mV = 1 °C
  float temp = voltaje * 100.0;

  return temp;
}


// =====================================================
// CONTROL DE TEMPERATURA
// =====================================================
void controlarTemperatura() {

  if (temperatura > 30.0) {

    // Temperatura superior a 30 °C
    // Ventilador al 100 %
    dutyVentilador = 255;

  } else {

    // Temperatura normal
    dutyVentilador = 0;
  }

  ledcWrite(CANAL_VENTILADOR, dutyVentilador);
}


// =====================================================
// CONTROL DE ILUMINACIÓN
// =====================================================
void controlarLuz() {

  /*
     Control proporcional inverso:

     Menor valor del LDR
            ↓
     Mayor iluminación
            ↓
     Mayor Duty Cycle del LED
  */

  dutyLED = map(
    valorLDR,
    0,
    4095,
    255,
    0
  );

  // Limitar valores
  dutyLED = constrain(dutyLED, 0, 255);

  ledcWrite(CANAL_LED, dutyLED);
}


// =====================================================
// TELEMETRÍA
// =====================================================
void enviarTelemetria() {

  Serial.println();
  Serial.println("---------- TELEMETRIA ----------");

  Serial.print("Temperatura: ");
  Serial.print(temperatura);
  Serial.println(" °C");

  Serial.print("LDR: ");
  Serial.println(valorLDR);

  Serial.print("Ventilador Duty: ");
  Serial.print(dutyVentilador);
  Serial.println(" / 255");

  Serial.print("LED Duty: ");
  Serial.print(dutyLED);
  Serial.println(" / 255");

  Serial.println("--------------------------------");
}


// =====================================================
// TELEMETRÍA JSON
// =====================================================
void enviarJSON() {

  Serial.print("{");

  Serial.print("\"temperatura\":");
  Serial.print(temperatura, 2);

  Serial.print(",\"ldr\":");
  Serial.print(valorLDR);

  Serial.print(",\"ventilador\":");
  Serial.print(dutyVentilador);

  Serial.print(",\"led\":");
  Serial.print(dutyLED);

  Serial.println("}");
}


// =====================================================
// PROCESAMIENTO DE COMANDOS
// =====================================================
void procesarComando() {

  if (Serial.available()) {

    String comando = Serial.readStringUntil('\n');

    comando.trim();

    if (comando == "TEMP") {

      Serial.print("Temperatura: ");
      Serial.print(temperatura);
      Serial.println(" °C");

    }

    else if (comando == "LDR") {

      Serial.print("Valor LDR: ");
      Serial.println(valorLDR);

    }

    else if (comando == "STATUS") {

      enviarTelemetria();

    }

    else if (comando == "JSON") {

      enviarJSON();

    }

    else if (comando == "FAN ON") {

      dutyVentilador = 255;

      ledcWrite(
        CANAL_VENTILADOR,
        dutyVentilador
      );

      Serial.println("Ventilador: ON");

    }

    else if (comando == "FAN OFF") {

      dutyVentilador = 0;

      ledcWrite(
        CANAL_VENTILADOR,
        dutyVentilador
      );

      Serial.println("Ventilador: OFF");

    }

    else if (comando == "LED OFF") {

      dutyLED = 0;

      ledcWrite(
        CANAL_LED,
        dutyLED
      );

      Serial.println("LED: OFF");

    }

    else {

      Serial.println("Comando no reconocido.");
      Serial.println("Comandos disponibles:");
      Serial.println("TEMP");
      Serial.println("LDR");
      Serial.println("STATUS");
      Serial.println("JSON");
      Serial.println("FAN ON");
      Serial.println("FAN OFF");
      Serial.println("LED OFF");
    }
  }
}


// =====================================================
// LOOP PRINCIPAL
// =====================================================
void loop() {

  // Leer sensores
  temperatura = leerTemperatura();
  valorLDR = analogRead(PIN_LDR);

  // Ejecutar controles
  controlarTemperatura();
  controlarLuz();

  // Procesar comandos del usuario
  procesarComando();

  // Mostrar información periódicamente
  enviarJSON();

  delay(1000);
}